# Loop Analysis During Training

This notebook monitors and analyzes whether agents get stuck in action loops **during training** (not just evaluation). Understanding loop behavior during training is crucial because:

- **Loops can waste training time**: If agents repeatedly execute the same sequence of actions, they're not exploring new states or learning efficiently
- **Loops may indicate learning issues**: Persistent loops in late training episodes might suggest the agent has converged to a suboptimal policy
- **Early detection helps**: Identifying loop patterns can guide improvements to exploration strategies, reward shaping, or hyperparameters

## Loop Definition

We use a **strict definition** of loops to focus on meaningful repetitive behavior:

**A loop is detected when:**
- A pattern of **at least 2 actions** is identified
- This pattern is repeated **more than 3 times** (i.e., at least 4 total occurrences)

**Examples:**
- Pattern `[up, down]` appearing 4 times: `up, down, up, down, up, down, up, down` → **LOOP DETECTED** (2-action pattern repeated 4 times)
- Pattern `[up, down]` appearing 3 times: `up, down, up, down, up, down` → **NOT A LOOP** (only 3 repetitions, need > 3)
- Pattern `[left, right, up]` appearing 5 times → **LOOP DETECTED** (3-action pattern repeated 5 times)

This strict definition helps us identify genuinely problematic repetitive behavior rather than short-term exploration patterns.

## Analysis Objectives

This notebook will:
1. Train agents using the standard training pipeline with action masking
2. Track action sequences in real-time during training
3. Detect loop patterns according to the strict definition above
4. Analyze when, how often, and for how long loops occur across episodes
5. Visualize loop statistics and correlate them with learning progress
6. Provide insights on whether loop-breaking mechanisms are needed in the main training pipeline


## Environment Configuration

This notebook uses a simplified **EmpathicGridEnv** configuration:

### Grid Schema

```
    0   1   2
  ┌───┬───┬───┐
0 │   │   │   │  A = Agent (blue circle)
  ├───┼───┼───┤  R = Resource (green square)
1 │   │ A │ R │  Vision radius = 2 (agent sees entire 3×3 grid)
  ├───┼───┼───┤
2 │   │   │   │
  └───┴───┴───┘
```

**Observation Window** (centered on agent):
```
    -1  -1  -1  -1  -1  -1  -1  -1  -1
    -1  -1  -1  -1  -1  -1  -1  -1  -1
    -1  -1   0   0   0  -1  -1  -1  -1
    -1  -1   0   2   1  -1  -1  -1  -1  ← Agent (2) sees Resource (1)
    -1  -1   0   0   0  -1  -1  -1  -1
    -1  -1  -1  -1  -1  -1  -1  -1  -1
    -1  -1  -1  -1  -1  -1  -1  -1  -1
    -1  -1  -1  -1  -1  -1  -1  -1  -1
    -1  -1  -1  -1  -1  -1  -1  -1  -1
```
*Note: Only the 3×3 grid is valid; surrounding -1 values represent out-of-bounds*

### Environment Parameters
- **Grid Size**: 3×3 (9 cells total)
- **Number of Agents**: 1 (single-agent scenario)
- **Number of Resources**: 1
- **Vision Radius**: 2 (agent can see the entire grid - 5×5 observation window centered on agent)
- **Max Steps per Episode**: 50

### Reward Structure
- **Step Cost**: -0.01 (penalty for each step taken, encourages efficiency)
- **Exploit Reward**: +1.0 (reward for consuming a resource)
- **Empathic Penalty**: -2.0 (penalty when consuming while more satisfied than nearby agents)

### Observation Space
Each agent receives a **local observation window** of shape `(5, 5, 3)`:
- **Channel 0 (Terrain)**: 
  - `-1` = out of bounds
  - `0` = empty cell
  - `1` = resource cell
  - `2` = agent cell (self at center)
- **Channel 1 (Other Agents' Satisfaction)**: Average satisfaction of other agents in view (0-1)
- **Channel 2 (Own Satisfaction)**: Agent's own satisfaction level (0-1, broadcasted)

### Action Space
- **4 discrete actions**: 0=Up, 1=Down, 2=Left, 3=Right
- **Action Masking**: Invalid actions (moving into walls) are prevented using `get_valid_actions()`

### Resource Consumption
- Resources are **automatically consumed** when an agent moves onto a resource cell
- After consumption, the resource **always respawns** at a random empty location

---

## Agent Configuration

### Agent Type: DQN (Deep Q-Network)

The notebook uses a **DQNAgent** with the following architecture and hyperparameters:

#### Network Architecture
- **Input**: Flattened observation (5×5×3 = 75 features)
- **Hidden Layers**: 
  - Fully connected layer: 75 → 64 (ReLU activation)
  - Fully connected layer: 64 → 64 (ReLU activation)
- **Output**: Q-values for 4 actions

#### Training Hyperparameters
- **Learning Rate**: 0.001
- **Discount Factor (γ)**: 0.99
- **Epsilon Start**: 1.0 (100% exploration at start)
- **Epsilon Decay**: 0.995 (per episode)
- **Epsilon Minimum**: 0.1 (maintains 10% exploration)
- **Batch Size**: 16
- **Target Network Update**: Every 20 steps

#### Agent Features
- **Experience Replay**: Stores and samples from past experiences
- **Target Network**: Stable Q-value targets for learning
- **Action Masking Support**: Filters invalid actions during action selection
- **Satisfaction Tracking**: Maintains meal history and calculates satisfaction/emotion

#### Satisfaction & Emotion Calculation
- **Beta (β)**: 0.5 (weight of last meal vs. history average)
- **Memory Size**: 10 (meal history buffer)
- **Threshold**: 0.5 (consumption rate for neutral emotion)
- **Smoothing**: Linear (maps consumption rate to emotion signal [-1, 1])


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from env_2d import EmpathicGridEnv
from agent_2d import Agent, DQNAgent

# Set random seed for reproducibility
np.random.seed(42)

print("Imports completed successfully!")


Imports completed successfully!


In [2]:
# GPU Test - Check if GPU is available and verify DQN agent uses it
import torch

print("="*60)
print("GPU AVAILABILITY CHECK")
print("="*60)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.current_device()}")
else:
    print("Using CPU (CUDA not available)")
print()

# Quick test: Create a DQN agent and verify device assignment
print("Testing DQN Agent GPU Usage...")
print("-"*60)

# Create a simple test environment
test_env = EmpathicGridEnv(
    grid_size=3,
    num_agents=1,
    num_resources=1,
    vision_radius=2,
    exploit_reward=1.0,
    empathic_penalty=-1.0,
    step_cost=-0.01,
    max_steps=50
)

# Create a DQN agent (should print device info)
test_agent = DQNAgent(
    agent_id=0,
    state_size=test_env.observation_space.shape,
    action_size=test_env.action_space.n,
    learning_rate=0.001,
    gamma=0.99,
    epsilon=1.0,
    epsilon_decay=0.995,
    epsilon_min=0.1,
    batch_size=16,
    hidden_size=64,
    update_target_every=20
)

# Verify network is on correct device
policy_device = next(test_agent.policy_network.parameters()).device
print(f"Policy network device: {policy_device}")
print(f"Target network device: {next(test_agent.target_network.parameters()).device}")

if policy_device.type == 'cuda':
    print("✓ SUCCESS: Agent is using GPU!")
else:
    print("ℹ INFO: Agent is using CPU (this is fine if GPU is not available)")

print()
print("="*60)

GPU AVAILABILITY CHECK
CUDA available: True
GPU device: NVIDIA GeForce RTX 4050 Laptop GPU
CUDA version: 12.1
Number of GPUs: 1
Current GPU: 0

Testing DQN Agent GPU Usage...
------------------------------------------------------------
Agent 0 using device: cuda
  GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Policy network device: cuda:0
Target network device: cuda:0
✓ SUCCESS: Agent is using GPU!



In [3]:
# Quick Training Test - Verify GPU is used during training
import time

print("="*60)
print("QUICK TRAINING TEST")
print("="*60)

# Use the test environment and agent from above
env_test = test_env
agent_test = test_agent
env_test.agents = [agent_test.agent]

# Run a few episodes to verify training works with GPU
print("Running 5 quick training episodes to verify GPU usage...")
print("-"*60)

episode_rewards = []
start_time = time.time()

for episode in range(5):
    obs, _ = env_test.reset(seed=42 + episode)
    agent_test.reset(observation=obs[0])
    
    episode_reward = 0.0
    for step in range(20):  # Short episodes
        valid_actions = env_test.get_valid_actions(0)
        action = agent_test.select_action(obs[0], training=True, valid_actions=valid_actions)
        next_obs, rewards, done, truncated, info = env_test.step([action])
        
        valid_actions_next = env_test.get_valid_actions(0)
        agent_test.step(next_obs[0], rewards[0], done, valid_actions=valid_actions_next)
        
        episode_reward += float(rewards[0])
        obs = next_obs
        
        if done:
            break
    
    episode_rewards.append(episode_reward)
    print(f"Episode {episode+1}: Reward = {episode_reward:.2f}, Steps = {step+1}")

elapsed_time = time.time() - start_time
print()
print(f"Total time for 5 episodes: {elapsed_time:.3f} seconds")
print(f"Average reward: {np.mean(episode_rewards):.2f}")
print()

# Check that replay buffer has samples and verify device of sampled tensors
if len(agent_test.memory) > 0:
    states, actions, rewards, next_states, dones = agent_test.memory.sample(
        min(4, len(agent_test.memory)), 
        agent_test.device
    )
    print(f"Replay buffer has {len(agent_test.memory)} experiences")
    print(f"Sampled batch - states device: {states.device}")
    print(f"Sampled batch - actions device: {actions.device}")
    print()
    
    if states.device.type == 'cuda':
        print("✓ SUCCESS: Training batches are on GPU!")
    else:
        print("ℹ INFO: Training batches are on CPU (GPU may not be available)")

print("="*60)
print("GPU test completed!")
print("="*60)

QUICK TRAINING TEST
Running 5 quick training episodes to verify GPU usage...
------------------------------------------------------------
Episode 1: Reward = 0.80, Steps = 20
Episode 2: Reward = 0.80, Steps = 20
Episode 3: Reward = 1.80, Steps = 20
Episode 4: Reward = 5.80, Steps = 20
Episode 5: Reward = 0.80, Steps = 20

Total time for 5 episodes: 1.142 seconds
Average reward: 2.00

Replay buffer has 95 experiences
Sampled batch - states device: cuda:0
Sampled batch - actions device: cuda:0

✓ SUCCESS: Training batches are on GPU!
GPU test completed!


In [4]:
class LoopDetector:
    """
    Detects repeating action patterns (loops) in agent behavior.
    
    Loop definition: A pattern of at least 2 actions that is repeated more than 3 times
    (i.e., the pattern must appear at least 4 times total).
    """
    
    def __init__(self, max_pattern_length=6, min_repetitions=4):
        """
        Args:
            max_pattern_length: Maximum length of pattern to check for (e.g., 6 = check patterns up to 6 actions)
            min_repetitions: Minimum number of times pattern must repeat to count as a loop (default: 4 = "more than 3 times")
        """
        self.max_pattern_length = max_pattern_length
        self.min_repetitions = min_repetitions  # Must be > 3 to meet "more than 3 times" requirement
        self.action_history = deque(maxlen=100)  # Keep last 100 actions
        self.loop_events = []  # List of detected loop events
        
    def add_action(self, action, step, episode):
        """
        Add a new action and check for loops.
        
        For a 2-action pattern repeated 4 times, we need at least 8 actions in history.
        """
        self.action_history.append(action)
        
        # Minimum needed: 2 actions (pattern) * 4 repetitions = 8 actions
        if len(self.action_history) < 8:  # Need at least 8 actions to detect a 2-action loop with 4+ repetitions
            return None
        
        loop_info = self._detect_loop(step, episode)
        return loop_info
    
    def _detect_loop(self, step, episode):
        """Detect if current action history contains a loop."""
        actions = list(self.action_history)
        
        # Check for loops of different lengths (2, 3, 4, etc.)
        for pattern_len in range(2, min(self.max_pattern_length + 1, len(actions) // self.min_repetitions + 1)):
            # Extract last few actions as potential pattern
            if len(actions) < pattern_len * self.min_repetitions:
                continue
                
            last_pattern = actions[-pattern_len:]
            
            # Check if this pattern repeats enough times
            repetitions = 1
            for i in range(2, len(actions) // pattern_len + 1):
                start_idx = len(actions) - i * pattern_len
                if start_idx < 0:
                    break
                candidate_pattern = actions[start_idx:start_idx + pattern_len]
                if candidate_pattern == last_pattern:
                    repetitions += 1
                else:
                    break
            
            if repetitions >= self.min_repetitions:
                # Found a loop!
                loop_event = {
                    'episode': episode,
                    'step': step,
                    'pattern': tuple(last_pattern),
                    'pattern_length': pattern_len,
                    'repetitions': repetitions,
                    'actions_in_loop': repetitions * pattern_len
                }
                self.loop_events.append(loop_event)
                return loop_event
        
        return None
    
    def get_statistics(self):
        """Get statistics about detected loops."""
        if not self.loop_events:
            return {
                'total_loops': 0,
                'loops_by_pattern_length': {},
                'avg_repetitions': 0,
                'loops_by_episode': {}
            }
        
        loops_by_pattern = {}
        loops_by_episode = {}
        total_repetitions = 0
        
        for event in self.loop_events:
            # Count by pattern length
            pl = event['pattern_length']
            loops_by_pattern[pl] = loops_by_pattern.get(pl, 0) + 1
            
            # Count by episode
            ep = event['episode']
            loops_by_episode[ep] = loops_by_episode.get(ep, 0) + 1
            
            total_repetitions += event['repetitions']
        
        return {
            'total_loops': len(self.loop_events),
            'loops_by_pattern_length': loops_by_pattern,
            'avg_repetitions': total_repetitions / len(self.loop_events),
            'loops_by_episode': loops_by_episode,
            'all_events': self.loop_events
        }
    
    def reset(self):
        """Reset the detector for a new training run."""
        self.action_history.clear()
        self.loop_events = []


print("LoopDetector class created!")


LoopDetector class created!


In [5]:
def train_multi_agent_with_loop_detection(env, num_episodes=200, max_steps_per_episode=50, seed=42):
    """
    Train multiple DQN agents (5 agents) and detect loops during training for each agent.
    
    Returns:
        all_episode_rewards: List of lists - rewards per episode per agent
        all_episode_steps: List of lists - steps per episode per agent
        all_loop_statistics: List of dictionaries - loop statistics per agent
        all_loop_detectors: List of LoopDetector instances - one per agent
        avg_episode_rewards: List - average rewards across agents per episode
    """
    np.random.seed(seed)
    num_agents = env.num_agents
    
    # Create DQN agents with TUNED hyperparameters to reduce loops:
    # - Higher learning rate (0.002) for faster learning
    # - Slower epsilon decay (0.998) to maintain exploration longer
    # - Higher epsilon min (0.2) to always have some randomness
    dqn_agents = []
    for agent_id in range(num_agents):
        agent = DQNAgent(
            agent_id=agent_id,
            state_size=env.observation_space.shape,
            action_size=env.action_space.n,
            learning_rate=0.002,      # Increased from 0.001
            gamma=0.99,
            epsilon=1.0,
            epsilon_decay=0.998,      # Slower decay (was 0.995)
            epsilon_min=0.2,          # Higher minimum (was 0.1)
            batch_size=16,
            hidden_size=64,
            update_target_every=20
        )
        dqn_agents.append(agent)
    
    # Link agents to environment
    env.agents = [agent.agent for agent in dqn_agents]
    
    # Create loop detector for each agent
    loop_detectors = [LoopDetector(max_pattern_length=6, min_repetitions=4) for _ in range(num_agents)]
    
    # Track metrics per agent
    all_episode_rewards = [[] for _ in range(num_agents)]
    all_episode_steps = [[] for _ in range(num_agents)]
    all_episode_loops = [[] for _ in range(num_agents)]
    
    for episode in range(num_episodes):
        obs, _ = env.reset(seed=seed + episode)
        for agent_idx, dqn_agent in enumerate(dqn_agents):
            dqn_agent.reset(observation=obs[agent_idx])
        
        episode_rewards = [0.0] * num_agents
        episode_loop_counts = [0] * num_agents
        
        for step in range(max_steps_per_episode):
            actions = []
            valid_actions_list = []
            
            # Collect actions from all agents with action masking
            for agent_idx, dqn_agent in enumerate(dqn_agents):
                valid_actions = env.get_valid_actions(agent_idx)
                valid_actions_list.append(valid_actions)
                action = dqn_agent.select_action(obs[agent_idx], training=True, valid_actions=valid_actions)
                actions.append(action)
                
                # Detect loops for this agent
                loop_event = loop_detectors[agent_idx].add_action(action, step, episode)
                if loop_event is not None:
                    episode_loop_counts[agent_idx] += 1
            
            # Step environment with all actions
            next_obs, rewards, done, truncated, info = env.step(actions)
            
            # Update all agents
            for agent_idx, dqn_agent in enumerate(dqn_agents):
                valid_actions_next = env.get_valid_actions(agent_idx)
                dqn_agent.step(next_obs[agent_idx], rewards[agent_idx], done, valid_actions=valid_actions_next)
                episode_rewards[agent_idx] += rewards[agent_idx]
            
            obs = next_obs
            if done:
                break
        
        # Record episode metrics for each agent
        for agent_idx in range(num_agents):
            all_episode_rewards[agent_idx].append(episode_rewards[agent_idx])
            all_episode_steps[agent_idx].append(step + 1)
            all_episode_loops[agent_idx].append(episode_loop_counts[agent_idx])
    
    # Compute statistics per agent
    all_loop_statistics = []
    for agent_idx in range(num_agents):
        loop_stats = loop_detectors[agent_idx].get_statistics()
        loop_stats['episode_loop_counts'] = all_episode_loops[agent_idx]
        all_loop_statistics.append(loop_stats)
    
    # Compute average rewards across agents per episode
    avg_episode_rewards = [np.mean([all_episode_rewards[agent_idx][ep] for agent_idx in range(num_agents)]) 
                           for ep in range(num_episodes)]
    
    return all_episode_rewards, all_episode_steps, all_loop_statistics, loop_detectors, avg_episode_rewards


print("Multi-agent training function created (with tuned hyperparameters)!")

Multi-agent training function created (with tuned hyperparameters)!


In [6]:
def run_multi_grid_multi_agent_test(grid_configs, num_episodes=200, max_steps_per_episode=50, base_seed=42):
    """
    Run training tests across multiple grid configurations with 5 agents each.
    Average results across all configurations.
    
    Args:
        grid_configs: List of dicts, each containing grid parameters:
            - grid_size: int
            - num_resources: int
            - vision_radius: int
            - max_steps: int
            - loop_penalty: float (optional, default -0.05)
        num_episodes: Number of episodes per configuration
        max_steps_per_episode: Maximum steps per episode
        base_seed: Base seed for randomization (each config uses base_seed + config_idx)
    
    Returns:
        aggregated_results: Dictionary with averaged metrics
    """
    all_config_results = []
    
    print(f"Running tests on {len(grid_configs)} different grid configurations...")
    print("="*60)
    
    for config_idx, config in enumerate(grid_configs):
        seed = base_seed + config_idx
        loop_penalty = config.get('loop_penalty', -0.05)  # Default to -0.05
        print(f"\nConfiguration {config_idx + 1}/{len(grid_configs)}: Grid {config['grid_size']}x{config['grid_size']}, "
              f"{config['num_resources']} resources, vision radius {config['vision_radius']}, loop_penalty={loop_penalty}")
        
        # Create environment for this configuration WITH LOOP PENALTY
        env = EmpathicGridEnv(
            grid_size=config['grid_size'],
            num_agents=5,  # 5 agents per configuration
            num_resources=config['num_resources'],
            vision_radius=config['vision_radius'],
            exploit_reward=1.0,
            empathic_penalty=-2.0,
            step_cost=-0.01,
            loop_penalty=loop_penalty,  # ENABLED: penalize loops
            max_steps=config['max_steps']
        )
        
        # Train on this configuration
        all_rewards, all_steps, all_loop_stats, all_detectors, avg_rewards = \
            train_multi_agent_with_loop_detection(
                env, 
                num_episodes=num_episodes,
                max_steps_per_episode=max_steps_per_episode,
                seed=seed
            )
        
        # Aggregate loop statistics across all 5 agents
        total_loops_all_agents = sum(stats['total_loops'] for stats in all_loop_stats)
        avg_repetitions_all_agents = np.mean([stats['avg_repetitions'] for stats in all_loop_stats if stats['total_loops'] > 0] or [0])
        
        # Count episodes with loops (any agent)
        episodes_with_loops = set()
        for stats in all_loop_stats:
            episodes_with_loops.update(stats['loops_by_episode'].keys())
        
        # Aggregate episode loop counts across agents
        episode_loop_counts_aggregated = []
        for ep in range(num_episodes):
            total_loops_this_episode = sum(stats['episode_loop_counts'][ep] for stats in all_loop_stats)
            episode_loop_counts_aggregated.append(total_loops_this_episode)
        
        config_result = {
            'config': config,
            'all_episode_rewards': all_rewards,  # Per-agent rewards
            'avg_episode_rewards': avg_rewards,  # Averaged across agents
            'all_episode_steps': all_steps,
            'all_loop_statistics': all_loop_stats,
            'total_loops_all_agents': total_loops_all_agents,
            'avg_repetitions_all_agents': avg_repetitions_all_agents,
            'episodes_with_loops': len(episodes_with_loops),
            'episode_loop_counts': episode_loop_counts_aggregated,
            'loops_by_pattern_length': {},  # Aggregate across agents
        }
        
        # Aggregate loops by pattern length across all agents
        for stats in all_loop_stats:
            for pattern_len, count in stats['loops_by_pattern_length'].items():
                config_result['loops_by_pattern_length'][pattern_len] = \
                    config_result['loops_by_pattern_length'].get(pattern_len, 0) + count
        
        all_config_results.append(config_result)
        print(f"  Total loops across all 5 agents: {total_loops_all_agents}")
        print(f"  Episodes with at least one loop: {len(episodes_with_loops)}/{num_episodes} ({len(episodes_with_loops)/num_episodes*100:.1f}%)")
    
    # Average results across all configurations
    print("\n" + "="*60)
    print("Averaging results across all configurations...")
    
    # Average episode rewards across configurations
    num_episodes_check = len(all_config_results[0]['avg_episode_rewards'])
    avg_rewards_across_configs = [np.mean([result['avg_episode_rewards'][ep] 
                                           for result in all_config_results]) 
                                  for ep in range(num_episodes_check)]
    
    # Average loop statistics across configurations
    avg_total_loops = np.mean([result['total_loops_all_agents'] for result in all_config_results])
    avg_avg_repetitions = np.mean([result['avg_repetitions_all_agents'] 
                                   for result in all_config_results 
                                   if result['avg_repetitions_all_agents'] > 0] or [0])
    
    avg_episodes_with_loops = np.mean([result['episodes_with_loops'] for result in all_config_results])
    avg_episodes_with_loops_pct = (avg_episodes_with_loops / num_episodes_check) * 100
    
    # Average episode loop counts
    avg_episode_loop_counts = [np.mean([result['episode_loop_counts'][ep] 
                                        for result in all_config_results]) 
                               for ep in range(num_episodes_check)]
    
    # Aggregate loops by pattern length across configurations
    avg_loops_by_pattern = {}
    all_pattern_lengths = set()
    for result in all_config_results:
        all_pattern_lengths.update(result['loops_by_pattern_length'].keys())
    for pattern_len in all_pattern_lengths:
        avg_loops_by_pattern[pattern_len] = np.mean([result['loops_by_pattern_length'].get(pattern_len, 0) 
                                                     for result in all_config_results])
    
    aggregated_results = {
        'all_config_results': all_config_results,
        'avg_episode_rewards': avg_rewards_across_configs,
        'avg_total_loops': avg_total_loops,
        'avg_avg_repetitions': avg_avg_repetitions,
        'avg_episodes_with_loops': avg_episodes_with_loops,
        'avg_episodes_with_loops_pct': avg_episodes_with_loops_pct,
        'avg_episode_loop_counts': avg_episode_loop_counts,
        'avg_loops_by_pattern_length': avg_loops_by_pattern,
        'num_configurations': len(grid_configs),
        'num_episodes': num_episodes_check
    }
    
    return aggregated_results


print("Multi-grid test function created (with loop penalty support)!")

Multi-grid test function created (with loop penalty support)!


---

## Understanding the Two Experiments in This Notebook

This notebook contains **two separate experiments** that analyze loop behavior under different conditions. Here's a clear breakdown:

---

### EXPERIMENT 1: Multi-Agent Simulation (Cells 9-12)

```
┌─────────────────────────────────────────────────────────────────┐
│                         5×5 GRID                                │
│  ┌───┬───┬───┬───┬───┐                                         │
│  │ A1│   │ R │   │ A2│   A1-A5 = 5 Agents (competing)          │
│  ├───┼───┼───┼───┼───┤   R = Resources (6 total)               │
│  │   │ R │   │ A3│   │                                         │
│  ├───┼───┼───┼───┼───┤   Each agent has its OWN DQN brain      │
│  │ R │   │   │ R │   │   They learn SIMULTANEOUSLY              │
│  ├───┼───┼───┼───┼───┤   They COMPETE for same resources       │
│  │   │ A4│ R │   │   │                                         │
│  ├───┼───┼───┼───┼───┤                                         │
│  │ A5│   │   │ R │   │                                         │
│  └───┴───┴───┴───┴───┘                                         │
└─────────────────────────────────────────────────────────────────┘
```

**What happens:**
- 5 agents with 5 separate neural networks
- They share the environment but compete for 6 resources
- Agents can BLOCK each other (collision avoidance)
- More chaotic → more opportunities for loops (stuck behind other agents)
- Results are AVERAGED across all 5 agents

**Why loops emerge:**
- Agent A wants to go right, but Agent B is blocking → A oscillates left-right
- Competition for resources creates congestion zones
- 5 brains learning different strategies simultaneously

---

### EXPERIMENT 2: Single-Agent Simulation (Cells 14-16)

```
┌─────────────────────────────────────────────────────────────────┐
│                         3×3 GRID                                │
│           ┌───┬───┬───┐                                        │
│           │   │   │   │   A = 1 Agent (alone)                  │
│           ├───┼───┼───┤   R = 1 Resource                       │
│           │   │ A │ R │                                        │
│           ├───┼───┼───┤   Agent has the ENTIRE grid to itself  │
│           │   │   │   │   No competition, no blocking          │
│           └───┴───┴───┘                                        │
└─────────────────────────────────────────────────────────────────┘
```

**What happens:**
- 1 agent with 1 neural network
- Tiny grid (3×3 = 9 cells only)
- Only 1 resource to find
- Agent can see EVERYTHING (vision radius covers whole grid)
- Much simpler learning task

**Why loops emerge:**
- No obstacle to blame → loops come purely from the policy
- Small grid = limited movement options
- "Lazy" policy: if resource is at (2,1) and agent at (1,1), optimal is just "right"
- But if epsilon is low and Q-values are wrong, agent oscillates

---

### Key Differences Summary

| Aspect | Experiment 1 (Multi-Agent) | Experiment 2 (Single-Agent) |
|--------|---------------------------|----------------------------|
| Grid size | 5×5 (25 cells) | 3×3 (9 cells) |
| Agents | 5 (competing) | 1 (alone) |
| Resources | 6 | 1 |
| Complexity | High (multi-agent dynamics) | Low (simple navigation) |
| Loop cause | Congestion + policy | Pure policy |
| Episodes | 100 | 200 |

---

### What to Look For in the Results

**Experiment 1 (Multi-Agent):**
- Loops should be HIGH because of agent collisions and competition
- Rewards may be NEGATIVE because of empathic penalties (taking resources when others need them)
- Learning is HARDER because the environment is non-stationary (other agents are also learning)

**Experiment 2 (Single-Agent):**
- Loops should be LOWER (no external obstacles)
- Rewards should be more POSITIVE (no competition)
- Learning should be CLEARER (stationary environment)

---

In [7]:
# Define different grid configurations to test
# TUNED CONFIGURATION to reduce loops:
# - More resources (6 instead of 3) to reduce competition/congestion
# - Loop penalty enabled (-0.05) to discourage repetitive behavior
grid_configs = [
    {'grid_size': 5, 'num_resources': 6, 'vision_radius': 2, 'max_steps': 100, 'loop_penalty': -0.05},
    {'grid_size': 5, 'num_resources': 6, 'vision_radius': 2, 'max_steps': 100, 'loop_penalty': -0.05}
]

# Run tests across all configurations
print("Starting multi-grid multi-agent test with 5 agents per configuration...")
print("TUNED HYPERPARAMETERS:")
print("  - Learning rate: 0.002 (was 0.001)")
print("  - Epsilon decay: 0.998 (was 0.995) - slower decay")
print("  - Epsilon min: 0.2 (was 0.1) - more exploration")
print("  - Num resources: 6 (was 3) - less congestion")
print("  - Loop penalty: -0.05 (was 0) - penalize loops")
print()
print(f"Number of configurations: {len(grid_configs)}")
print(f"Total agents: {len(grid_configs) * 5}")
print()

aggregated_results = run_multi_grid_multi_agent_test(
    grid_configs=grid_configs,
    num_episodes=100,
    max_steps_per_episode=50,
    base_seed=42
)

print("\n" + "="*60)
print("AGGREGATED RESULTS (Averaged across all configurations and agents)")
print("="*60)
print(f"Average total loops per configuration (across 5 agents): {aggregated_results['avg_total_loops']:.2f}")
print(f"Average repetitions per loop: {aggregated_results['avg_avg_repetitions']:.2f}")
print()
print("Average loops by pattern length:")
for pattern_len, count in sorted(aggregated_results['avg_loops_by_pattern_length'].items()):
    print(f"  {pattern_len}-action loops: {count:.2f} occurrences")
print()
print(f"Average episodes with at least one loop: {aggregated_results['avg_episodes_with_loops']:.1f}/{aggregated_results['num_episodes']}")
print(f"Percentage: {aggregated_results['avg_episodes_with_loops_pct']:.1f}%")

Starting multi-grid multi-agent test with 5 agents per configuration...
TUNED HYPERPARAMETERS:
  - Learning rate: 0.002 (was 0.001)
  - Epsilon decay: 0.998 (was 0.995) - slower decay
  - Epsilon min: 0.2 (was 0.1) - more exploration
  - Num resources: 6 (was 3) - less congestion
  - Loop penalty: -0.05 (was 0) - penalize loops

Number of configurations: 2
Total agents: 10

Running tests on 2 different grid configurations...

Configuration 1/2: Grid 5x5, 6 resources, vision radius 2, loop_penalty=-0.05
Agent 0 using device: cuda
  GPU: NVIDIA GeForce RTX 4050 Laptop GPU
  Total loops across all 5 agents: 1785
  Episodes with at least one loop: 92/100 (92.0%)

Configuration 2/2: Grid 5x5, 6 resources, vision radius 2, loop_penalty=-0.05
Agent 0 using device: cuda
  GPU: NVIDIA GeForce RTX 4050 Laptop GPU
  Total loops across all 5 agents: 1434
  Episodes with at least one loop: 91/100 (91.0%)

Averaging results across all configurations...

AGGREGATED RESULTS (Averaged across all config

In [ ]:
# Visualize averaged results
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Average reward learning curve (raw + smoothed)
raw_rewards = aggregated_results['avg_episode_rewards']
# Smooth with moving average (window of 20 episodes)
window_size = 20
smoothed_rewards = [np.mean(raw_rewards[max(0, i-window_size//2):min(len(raw_rewards), i+window_size//2+1)]) 
                    for i in range(len(raw_rewards))]

axes[0, 0].plot(raw_rewards, alpha=0.3, linewidth=1, color='lightblue', label='Raw (per episode)')
axes[0, 0].plot(smoothed_rewards, alpha=0.9, linewidth=2, color='darkblue', label=f'Smoothed (window={window_size})')
axes[0, 0].set_title('Average Episode Rewards (Averaged across 5 agents and all grid configs)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Average Reward')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[0, 0].legend()

# 2. Average loop occurrences per episode (raw + smoothed)
raw_loops = aggregated_results['avg_episode_loop_counts']
smoothed_loops = [np.mean(raw_loops[max(0, i-window_size//2):min(len(raw_loops), i+window_size//2+1)]) 
                  for i in range(len(raw_loops))]

axes[0, 1].plot(raw_loops, alpha=0.3, linewidth=1, color='lightcoral', label='Raw (per episode)')
axes[0, 1].plot(smoothed_loops, alpha=0.9, linewidth=2, color='darkorange', label=f'Smoothed (window={window_size})')
axes[0, 1].set_title('Average Loop Counts per Episode (Averaged across configs)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Average Number of Loops')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# 3. Loop distribution by pattern length (averaged)
pattern_lengths = sorted(aggregated_results['avg_loops_by_pattern_length'].keys())
pattern_counts = [aggregated_results['avg_loops_by_pattern_length'][pl] for pl in pattern_lengths]
axes[1, 0].bar(pattern_lengths, pattern_counts, color='coral', alpha=0.7)
axes[1, 0].set_title('Average Loops by Pattern Length', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Pattern Length (number of actions)')
axes[1, 0].set_ylabel('Average Number of Loops')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Loop distribution across training phases
num_episodes = aggregated_results['num_episodes']
early_phase = aggregated_results['avg_episode_loop_counts'][:num_episodes//3]
mid_phase = aggregated_results['avg_episode_loop_counts'][num_episodes//3:2*num_episodes//3]
late_phase = aggregated_results['avg_episode_loop_counts'][2*num_episodes//3:]

phases = ['Early\n(0-33%)', 'Mid\n(33-66%)', 'Late\n(66-100%)']
phase_means = [np.mean(early_phase), np.mean(mid_phase), np.mean(late_phase)]
axes[1, 1].bar(phases, phase_means, color=['lightblue', 'orange', 'lightcoral'], alpha=0.7)
axes[1, 1].set_title('Average Loops per Episode by Training Phase', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Average Loops per Episode')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('fig1_multi_agent_overview.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"Number of grid configurations tested: {aggregated_results['num_configurations']}")
print(f"Agents per configuration: 5")
print(f"Total agents: {aggregated_results['num_configurations'] * 5}")
print(f"Episodes per configuration: {aggregated_results['num_episodes']}")
print()
print(f"Average total loops per configuration (5 agents): {aggregated_results['avg_total_loops']:.2f}")
print(f"Average loops per agent per configuration: {aggregated_results['avg_total_loops']/5:.2f}")
print(f"Average repetitions per loop: {aggregated_results['avg_avg_repetitions']:.2f}")
print()
print(f"Average episodes with loops: {aggregated_results['avg_episodes_with_loops']:.1f}/{aggregated_results['num_episodes']} ({aggregated_results['avg_episodes_with_loops_pct']:.1f}%)")
print(f"Average loops per episode: {np.mean(aggregated_results['avg_episode_loop_counts']):.2f}")
print(f"Max average loops in a single episode: {np.max(aggregated_results['avg_episode_loop_counts']):.2f}")
print()
print("Learning trend analysis:")
early_rewards = np.mean(raw_rewards[:len(raw_rewards)//3])
mid_rewards = np.mean(raw_rewards[len(raw_rewards)//3:2*len(raw_rewards)//3])
late_rewards = np.mean(raw_rewards[2*len(raw_rewards)//3:])
print(f"  Early episodes (0-33%): average reward = {early_rewards:.3f}")
print(f"  Mid episodes (33-66%): average reward = {mid_rewards:.3f}")
print(f"  Late episodes (66-100%): average reward = {late_rewards:.3f}")
print(f"  Improvement (late - early): {late_rewards - early_rewards:.3f}")
if late_rewards > early_rewards:
    print(f"  ✓ Positive learning trend detected!")
else:
    print(f"  ✗ No clear improvement (may need more episodes or hyperparameter tuning)")

### Individual Configuration Analysis

Let's also visualize learning curves for each grid configuration separately to see if some configurations show clearer learning trends:


In [ ]:
# Plot individual configuration learning curves
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

colors = ['blue', 'green', 'orange', 'red', 'purple']
for idx, config_result in enumerate(aggregated_results['all_config_results']):
    config = config_result['config']
    config_rewards = config_result['avg_episode_rewards']
    
    # Smooth the rewards for this configuration
    smoothed_config_rewards = [np.mean(config_rewards[max(0, i-window_size//2):min(len(config_rewards), i+window_size//2+1)]) 
                               for i in range(len(config_rewards))]
    
    label = f"Grid {config['grid_size']}×{config['grid_size']}, {config['num_resources']} resources"
    ax.plot(smoothed_config_rewards, alpha=0.8, linewidth=2, color=colors[idx % len(colors)], label=label)

# Also plot the overall average
ax.plot(smoothed_rewards, alpha=1.0, linewidth=3, color='black', linestyle='--', label='Overall Average')

ax.set_title('Learning Curves per Grid Configuration (Smoothed, window=20)', fontsize=14, fontweight='bold')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Average Reward (across 5 agents)', fontsize=12)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.legend(loc='best', fontsize=10)
plt.tight_layout()
plt.savefig('fig2_multi_agent_per_config.png', dpi=150, bbox_inches='tight')
plt.show()

# Print learning trends for each configuration
print("\n" + "="*60)
print("LEARNING TRENDS BY CONFIGURATION")
print("="*60)
for idx, config_result in enumerate(aggregated_results['all_config_results']):
    config = config_result['config']
    config_rewards = config_result['avg_episode_rewards']
    
    early = np.mean(config_rewards[:len(config_rewards)//3])
    mid = np.mean(config_rewards[len(config_rewards)//3:2*len(config_rewards)//3])
    late = np.mean(config_rewards[2*len(config_rewards)//3:])
    improvement = late - early
    
    print(f"\nGrid {config['grid_size']}×{config['grid_size']} ({config['num_resources']} resources):")
    print(f"  Early: {early:.3f} | Mid: {mid:.3f} | Late: {late:.3f} | Improvement: {improvement:+.3f}")

In [10]:
def train_agent_with_loop_detection(env, num_episodes=200, max_steps_per_episode=50, seed=42):
    """
    Train a DQN agent and detect loops during training.
    
    Returns:
        episode_rewards: List of total rewards per episode
        episode_steps: List of steps per episode
        loop_statistics: Dictionary with loop detection statistics
        loop_detector: The LoopDetector instance used
    """
    np.random.seed(seed)
    
    # Create DQN agent with TUNED hyperparameters to reduce loops:
    # - Higher learning rate (0.002) for faster learning
    # - Slower epsilon decay (0.998) to maintain exploration longer
    # - Higher epsilon min (0.2) to always have some randomness
    dqn_agent = DQNAgent(
        agent_id=0,
        state_size=env.observation_space.shape,
        action_size=env.action_space.n,
        learning_rate=0.002,      # Increased from 0.001
        gamma=0.99,
        epsilon=1.0,
        epsilon_decay=0.998,      # Slower decay (was 0.995)
        epsilon_min=0.2,          # Higher minimum (was 0.1)
        batch_size=16,
        hidden_size=64,
        update_target_every=20
    )
    
    # Link agent to environment
    env.agents = [dqn_agent.agent]
    
    # Create loop detector with strict definition: pattern of 2+ actions repeated more than 3 times (≥4 times)
    loop_detector = LoopDetector(max_pattern_length=6, min_repetitions=4)
    
    episode_rewards = []
    episode_steps = []
    episode_loops = []  # Number of loops detected per episode
    
    for episode in range(num_episodes):
        obs, _ = env.reset(seed=seed + episode)
        dqn_agent.reset(observation=obs[0])
        
        episode_reward = 0
        episode_loop_count = 0
        
        for step in range(max_steps_per_episode):
            # Get valid actions for action masking
            valid_actions = env.get_valid_actions(0)
            
            # Select action with action masking
            action = dqn_agent.select_action(obs[0], training=True, valid_actions=valid_actions)
            
            # Detect loops
            loop_event = loop_detector.add_action(action, step, episode)
            if loop_event is not None:
                episode_loop_count += 1
                # Optionally print loop detection (commented out to avoid spam)
                # print(f"Episode {episode}, Step {step}: Detected {loop_event['pattern_length']}-action loop "
                #       f"{loop_event['pattern']} repeated {loop_event['repetitions']} times")
            
            # Step environment
            next_obs, rewards, done, truncated, info = env.step([action])
            
            # Get valid actions for next state
            valid_actions_next = env.get_valid_actions(0)
            
            # Update agent with action masking
            dqn_agent.step(next_obs[0], rewards[0], done, valid_actions=valid_actions_next)
            
            episode_reward += rewards[0]
            
            obs = next_obs
            if done:
                break
        
        episode_rewards.append(episode_reward)
        episode_steps.append(step + 1)
        episode_loops.append(episode_loop_count)
    
    loop_statistics = loop_detector.get_statistics()
    loop_statistics['episode_loop_counts'] = episode_loops
    
    return episode_rewards, episode_steps, loop_statistics, loop_detector


print("Training function with loop detection created (with tuned hyperparameters)!")

Training function with loop detection created (with tuned hyperparameters)!


In [11]:
# Create environment with TUNED parameters
# - Loop penalty enabled (-0.05) to discourage repetitive behavior
env_loop_test = EmpathicGridEnv(
    grid_size=3,
    num_agents=1,
    num_resources=1,
    vision_radius=2,
    exploit_reward=1.0,
    empathic_penalty=-1.0,
    step_cost=-0.01,
    loop_penalty=-0.05,  # ENABLED: penalize loops
    max_steps=50
)

print("Environment created with TUNED parameters:")
print(f"Grid size: 3x3")
print(f"Vision radius: 2 (agent can see whole grid)")
print(f"Loop penalty: -0.05 (penalize repetitive behavior)")
print()
print("Agent hyperparameters:")
print(f"  - Learning rate: 0.002 (was 0.001)")
print(f"  - Epsilon decay: 0.998 (was 0.995)")
print(f"  - Epsilon min: 0.2 (was 0.1)")
print()
print(f"Training for 200 episodes...")
print()

# Train and detect loops
episode_rewards, episode_steps, loop_stats, detector = train_agent_with_loop_detection(
    env_loop_test,
    num_episodes=200,
    max_steps_per_episode=50,
    seed=42
)

print("Training completed!")
print()
print("="*60)
print("LOOP DETECTION RESULTS")
print("="*60)
print(f"Total loops detected: {loop_stats['total_loops']}")
print(f"Average repetitions per loop: {loop_stats['avg_repetitions']:.2f}")
print()
print("Loops by pattern length:")
for pattern_len, count in sorted(loop_stats['loops_by_pattern_length'].items()):
    print(f"  {pattern_len}-action loops: {count} occurrences")
print()
print(f"Total episodes with at least one loop: {len(loop_stats['loops_by_episode'])}")
print(f"Percentage of episodes with loops: {len(loop_stats['loops_by_episode']) / 200 * 100:.1f}%")

Environment created with TUNED parameters:
Grid size: 3x3
Vision radius: 2 (agent can see whole grid)
Loop penalty: -0.05 (penalize repetitive behavior)

Agent hyperparameters:
  - Learning rate: 0.002 (was 0.001)
  - Epsilon decay: 0.998 (was 0.995)
  - Epsilon min: 0.2 (was 0.1)

Training for 200 episodes...

Agent 0 using device: cuda
  GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Training completed!

LOOP DETECTION RESULTS
Total loops detected: 406
Average repetitions per loop: 4.75

Loops by pattern length:
  2-action loops: 406 occurrences

Total episodes with at least one loop: 95
Percentage of episodes with loops: 47.5%


In [ ]:
# Visualize loop occurrences over training
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Loops per episode
episode_loop_counts = loop_stats['episode_loop_counts']
axes[0, 0].plot(episode_loop_counts, color='red', alpha=0.7, linewidth=1)
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Number of Loops Detected')
axes[0, 0].set_title('Loop Occurrences Per Episode During Training')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(y=0, color='black', linestyle='--', linewidth=0.5)

# Add moving average
if len(episode_loop_counts) > 10:
    window = 20
    moving_avg = np.convolve(episode_loop_counts, np.ones(window)/window, mode='valid')
    axes[0, 0].plot(range(window-1, len(episode_loop_counts)), moving_avg, 
                    color='darkred', linewidth=2, label=f'Moving Average ({window} episodes)')
    axes[0, 0].legend()

# Plot 2: Loops by pattern length
pattern_lengths = sorted(loop_stats['loops_by_pattern_length'].keys())
pattern_counts = [loop_stats['loops_by_pattern_length'][pl] for pl in pattern_lengths]
axes[0, 1].bar(pattern_lengths, pattern_counts, color='orange', alpha=0.7)
axes[0, 1].set_xlabel('Pattern Length (number of actions)')
axes[0, 1].set_ylabel('Number of Loop Occurrences')
axes[0, 1].set_title('Loop Distribution by Pattern Length')
axes[0, 1].grid(True, alpha=0.3, axis='y')
for i, (pl, cnt) in enumerate(zip(pattern_lengths, pattern_counts)):
    axes[0, 1].text(pl, cnt, str(cnt), ha='center', va='bottom')

# Plot 3: Episode rewards (for context)
axes[1, 0].plot(episode_rewards, alpha=0.5, color='blue', linewidth=1)
# Smooth curve
window = 20
if len(episode_rewards) > window:
    smoothed = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
    axes[1, 0].plot(range(window-1, len(episode_rewards)), smoothed, 
                    color='darkblue', linewidth=2, label=f'Smoothed ({window} episodes)')
    axes[1, 0].legend()
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Total Episode Reward')
axes[1, 0].set_title('Learning Progress (Episode Rewards)')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Loop frequency over time (binning episodes)
bin_size = 20
num_bins = len(episode_loop_counts) // bin_size
if num_bins > 0:
    binned_loops = [np.mean(episode_loop_counts[i*bin_size:(i+1)*bin_size]) 
                    for i in range(num_bins)]
    x_bins = [i*bin_size + bin_size//2 for i in range(num_bins)]
    axes[1, 1].plot(x_bins, binned_loops, 'o-', color='red', linewidth=2, markersize=8)
    axes[1, 1].set_xlabel('Episode (bin centers)')
    axes[1, 1].set_ylabel('Average Loops per Episode')
    axes[1, 1].set_title(f'Loop Frequency Over Training ({bin_size}-episode bins)')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].axhline(y=0, color='black', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.savefig('fig3_single_agent_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAverage loops per episode: {np.mean(episode_loop_counts):.2f}")
print(f"Max loops in a single episode: {np.max(episode_loop_counts)}")
print(f"Episodes with zero loops: {sum(1 for x in episode_loop_counts if x == 0)} ({sum(1 for x in episode_loop_counts if x == 0)/len(episode_loop_counts)*100:.1f}%)")

In [ ]:
# Detailed analysis: Show some example loops
print("="*60)
print("EXAMPLE LOOP DETECTIONS")
print("="*60)
print()

all_events = loop_stats['all_events']
if all_events:
    # Show first 10 loops
    print("First 10 detected loops:")
    print()
    for i, event in enumerate(all_events[:10]):
        print(f"Loop {i+1}:")
        print(f"  Episode: {event['episode']}, Step: {event['step']}")
        print(f"  Pattern: {event['pattern']} ({event['pattern_length']} actions)")
        print(f"  Repetitions: {event['repetitions']} times")
        print(f"  Total actions in loop: {event['actions_in_loop']}")
        print()
    
    # Analyze when loops occur most
    early_episodes = [e for e in all_events if e['episode'] < 50]
    mid_episodes = [e for e in all_events if 50 <= e['episode'] < 150]
    late_episodes = [e for e in all_events if e['episode'] >= 150]
    
    print("Loop distribution across training:")
    print(f"  Early episodes (0-49): {len(early_episodes)} loops ({len(early_episodes)/len(all_events)*100:.1f}%)")
    print(f"  Mid episodes (50-149): {len(mid_episodes)} loops ({len(mid_episodes)/len(all_events)*100:.1f}%)")
    print(f"  Late episodes (150-199): {len(late_episodes)} loops ({len(late_episodes)/len(all_events)*100:.1f}%)")
else:
    print("No loops detected during training!")


EXAMPLE LOOP DETECTIONS

First 10 detected loops:

Loop 1:
  Episode: 6, Step: 37
  Pattern: (2, 3) (2 actions)
  Repetitions: 4 times
  Total actions in loop: 8

Loop 2:
  Episode: 13, Step: 30
  Pattern: (0, 1) (2 actions)
  Repetitions: 4 times
  Total actions in loop: 8

Loop 3:
  Episode: 13, Step: 31
  Pattern: (1, 0) (2 actions)
  Repetitions: 4 times
  Total actions in loop: 8

Loop 4:
  Episode: 13, Step: 32
  Pattern: (0, 1) (2 actions)
  Repetitions: 5 times
  Total actions in loop: 10

Loop 5:
  Episode: 16, Step: 9
  Pattern: (1, 0) (2 actions)
  Repetitions: 4 times
  Total actions in loop: 8

Loop 6:
  Episode: 16, Step: 10
  Pattern: (0, 1) (2 actions)
  Repetitions: 4 times
  Total actions in loop: 8

Loop 7:
  Episode: 16, Step: 11
  Pattern: (1, 0) (2 actions)
  Repetitions: 5 times
  Total actions in loop: 10

Loop 8:
  Episode: 17, Step: 35
  Pattern: (0, 1) (2 actions)
  Repetitions: 4 times
  Total actions in loop: 8

Loop 9:
  Episode: 17, Step: 36
  Pattern: (1

---

## Notebook Structure Summary

This notebook is organized into distinct sections that progressively build up the loop analysis pipeline:

### Section 1: Introduction & Configuration (Cells 1-2)
- **Purpose**: Define the problem (loop detection during training) and document the environment/agent configuration
- **Key content**: Loop definition (pattern of 2+ actions repeated 4+ times), grid schema, observation space, reward structure, DQN hyperparameters

### Section 2: Setup & GPU Verification (Cells 3-5)
- **Purpose**: Import dependencies and verify GPU availability for efficient training
- **Key content**: CUDA check, device assignment verification, quick training test to confirm GPU usage

### Section 3: Core Components (Cells 6-8)
- **Purpose**: Define the reusable classes and functions for loop detection and training
- **Components**:
  - `LoopDetector`: Detects repeating action patterns using a strict definition
  - `train_multi_agent_with_loop_detection()`: Trains multiple DQN agents while tracking loops
  - `run_multi_grid_multi_agent_test()`: Runs experiments across multiple grid configurations

### Section 4: Multi-Agent Experiments (Cells 9-13)
- **Purpose**: Run large-scale experiments with 5 agents across multiple grid configurations
- **Key outputs**: Aggregated statistics, learning curves, loop frequency analysis, per-configuration breakdowns

### Section 5: Single-Agent Deep Dive (Cells 14-17)
- **Purpose**: Detailed analysis on a simpler single-agent scenario for clearer insights
- **Key outputs**: Loop detection results, example loop patterns, temporal distribution of loops

---

## Why This Analysis Matters

Understanding loop behavior during training is critical for several reasons:

### 1. Training Efficiency
Loops represent **wasted computation**. When an agent repeatedly executes the same action sequence (e.g., `[left, right, left, right, ...]`), it:
- Consumes training steps without exploring new states
- Fills the replay buffer with redundant experiences
- Slows down convergence to optimal policies

### 2. Diagnosis of Learning Problems
Loop patterns can indicate:
- **Insufficient exploration**: Epsilon may be decaying too fast
- **Reward sparsity**: Agent doesn't receive enough signal to learn meaningful behavior
- **Local optima**: Agent has converged to a suboptimal policy
- **Action masking issues**: Valid actions may be too restrictive

### 3. Reward Shaping Validation
The main training pipeline (`main_2d.py`) includes a **loop penalty** parameter. This notebook validates:
- Whether loops are actually occurring during training
- How often and when they occur (early vs. late training)
- Whether the penalty magnitude is appropriate

### 4. Multi-Agent Dynamics
In multi-agent settings, loops can emerge from:
- **Collision avoidance**: Agents blocking each other's paths
- **Resource competition**: Agents oscillating around contested resources
- **Coordination failures**: Lack of implicit communication leading to repetitive patterns

### 5. Hyperparameter Tuning Guidance
The analysis provides insights for tuning:
- `epsilon_decay`: Slower decay if loops persist in late training
- `loop_penalty`: Increase magnitude if loops remain frequent
- `vision_radius`: Larger vision may help agents navigate more effectively
- `num_resources`: Resource density affects loop likelihood

---

## Key Findings Template

After running this notebook, document your findings:

1. **Loop Frequency**: What percentage of episodes contain loops?
2. **Temporal Distribution**: Do loops decrease as training progresses?
3. **Pattern Types**: Are loops mostly 2-action (back-and-forth) or longer patterns?
4. **Multi-Agent vs Single-Agent**: How does agent count affect loop occurrence?
5. **Configuration Sensitivity**: Which grid configurations show more loops?

These insights directly inform improvements to the main training pipeline.